<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 6 &nbsp;&middot;&nbsp; Sequences, RNNs, and LSTMs</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>How neural networks read sequences &mdash; the hidden state idea, why vanilla RNNs forget too quickly, how LSTMs fix this with gating, and a full end-to-end forecasting application on real atmospheric CO&#8322; data.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| Introduction | What RNNs and LSTMs are — in plain language |
| 6.1 Why Sequences Are Different | Order matters · Hidden state concept · Business examples |
| 6.2 Vanilla RNN | The reading-a-sentence analogy · Unrolling · `nn.RNN` · Vanishing gradients |
| 6.3 LSTM | Gates as selective memory · Cell state · RNN vs LSTM comparison · `nn.LSTM` |
| 6.4 Business Application: CO₂ Forecasting | Mauna Loa dataset · Sliding windows · Train · Forecast · Visualise |
| 6.5 Pipeline: Saving the Forecast Model | `ModelPipeline` for sequences · Save · Load · Forecast · Retrain |
| 6.6 ★ Bonus: Autoencoders | Encoder–bottleneck–decoder · Anomaly detection · Three hands-on exercises |

> **Dataset:** Mauna Loa CO₂ — built into `statsmodels`. No download, no account, no API key required.

---
## Before We Begin: What Are RNNs and LSTMs?

Every architecture you have built so far treats each input as a standalone snapshot. Feed a customer record to an FFN (Chapter 4) and it produces a prediction without knowing whether that customer has been with you for one month or ten years. Feed a product image to a CNN (Chapter 5) and it detects defects without knowing whether the previous twenty images on the same line were also defective.

That is fine when inputs really are independent. But many of the most valuable datasets in business are not independent — they are **sequences**: ordered streams where the past directly shapes what comes next.

Think of reading a sentence. You do not treat each word in isolation. You carry meaning forward as you read — each word is understood in the context of everything that came before it. "The bank was steep" means something very different from "The bank was closed" — and you can only tell the difference because you remember the surrounding words.

**Recurrent Neural Networks (RNNs)** were designed to do exactly this: process sequences by maintaining a running **hidden state** — a compact summary of everything seen so far — that is updated at each step. Think of it as the network's short-term memory.

**Long Short-Term Memory networks (LSTMs)** are an improved version of RNNs that solve a critical weakness: vanilla RNNs tend to forget information from early in a sequence before they reach the end. LSTMs fix this by adding a carefully controlled long-term memory channel that can hold information across hundreds of steps.

By the end of this chapter you will have trained an LSTM to forecast atmospheric CO₂ concentration — a real scientific dataset with a strong trend and a beautiful yearly seasonal pattern — and watched it predict the future from 40 years of history.

> **What you already know that applies here:**  
> The training loop, loss functions, optimisers, and `ModelPipeline` from Chapters 3–5 are all used unchanged. The only new idea is *how the model processes its input* — one step at a time, with memory.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 6 — Setup
# Run this cell first. All packages except statsmodels are already on Colab;
# statsmodels installs in ~10 seconds.
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q statsmodels dill

import copy, datetime, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
print('Setup complete.')

---
## Shared Training Helpers

These four helpers are identical to the versions introduced in Chapter 3 and reused in Chapters 4 and 5. They are copied here so this notebook is self-contained.

> **One addition:** `train_one_epoch` now calls `nn.utils.clip_grad_norm_` before the weight update. **Gradient clipping** is a standard best practice for RNNs and LSTMs — it prevents the rare but catastrophic case where gradients grow very large during backpropagation through long sequences. You will not notice it in normal training; it just quietly prevents instability.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared helpers — same contract as Chapters 3, 4, and 5.
# One addition: gradient clipping in train_one_epoch (best practice for RNNs).
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimiser, device):
    model.train()
    total = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # RNN best practice
        optimiser.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total += criterion(model(X_batch), y_batch).item()
    return total / len(loader)


def run_training(model, train_loader, val_loader, criterion, optimiser,
                 num_epochs, device, scheduler=None, val_loss_scheduler=None,
                 early_stopping=None, verbose_every=5):
    """Standard training loop. Returns (train_losses, val_losses)."""
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        tl = train_one_epoch(model, train_loader, criterion, optimiser, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tl)
        val_losses.append(vl)
        if scheduler:          scheduler.step()
        if val_loss_scheduler: val_loss_scheduler.step(vl)
        if verbose_every and (epoch % verbose_every == 0 or epoch == num_epochs - 1):
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if early_stopping and early_stopping.step(vl, model):
            print(f'  Early stopping at epoch {epoch}  (best val: {early_stopping.best_loss:.4f})')
            early_stopping.restore_best(model)
            break
    return train_losses, val_losses


class EarlyStopping:
    """Halt training when validation loss stops improving. Identical to Chapters 3–5."""
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_loss = float('inf'); self.counter = 0; self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_weights: model.load_state_dict(self.best_weights)


print('Helpers defined: train_one_epoch | evaluate | run_training | EarlyStopping')

---
# 6.1 Why Sequences Are Different

## Order matters

"Dog bites man" and "Man bites dog" contain exactly the same words. They are very different sentences. Order is meaning.

This is not a quirk of language. The same principle applies across all the sequential data that businesses care about:

| Data | Each step | Why order matters |
|------|-----------|-------------------|
| Daily sales | One day's revenue | Trend, seasonality, and momentum are temporal — yesterday's sales matter |
| Energy consumption | Hourly power reading | Morning and evening peaks repeat in predictable sequence |
| Customer clickstream | One page visit | The path taken so far determines the most likely next action |
| Sensor readings | One measurement | An anomaly is only visible as a deviation from recent history |

An FFN sees each input as a frozen snapshot. It has no way to carry information from one step to the next. To process a sequence of 60 readings, an FFN would have to receive all 60 at once as a single flat vector — which destroys the temporal relationships between them.

Recurrent networks were designed specifically to avoid this: they process sequences **one step at a time**, maintaining a **hidden state** that travels forward through the sequence like a traveller carrying notes from each stop.

## The hidden state — the central idea

Imagine reading a long paragraph by taking notes as you go. After each sentence you jot down a brief summary of everything you have read so far. When you reach the next sentence, you read it in the context of your notes — not just in isolation.

That is exactly what a recurrent network does. At every time step $t$:

1. It receives the **current input** $x_t$ (e.g. today's CO₂ reading)
2. It reads the **previous hidden state** $h_{t-1}$ (its notes so far)
3. It produces a **new hidden state** $h_t$ (updated notes)
4. Optionally, it produces an **output** $y_t$ (a prediction)

```
         x₁          x₂          x₃          x₄
         │           │           │           │
    ┌────▼────┐ h₁  ┌▼────────┐ h₂  ┌▼────────┐ h₃  ┌▼────────┐
    │  RNN    ├────►│  RNN    ├────►│  RNN    ├────►│  RNN    │
    │  cell   │     │  cell   │     │  cell   │     │  cell   │
    └────┬────┘     └────┬────┘     └────┬────┘     └────┬────┘
         │               │               │               │
         y₁              y₂              y₃              y₄
```

*Figure 6.1 — A recurrent network unrolled through four time steps. The same cell (same weights) is applied at every step. The hidden state $h_t$ carries information forward.*

> **Important:** The weights inside the RNN cell are **shared across all time steps**. The same transformation is applied at step 1 as at step 100. This is why an RNN can handle sequences of any length — there are no extra parameters for longer sequences.

### 📝 Exercise 6.1 — Sequence Problems in Your Domain

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.1
#
# Think of three sequential datasets from your own field or industry.
# For each one, fill in the three lines below.
#
# Example:
#   Sequence : Daily number of customer support tickets
#   Target   : Volume tomorrow (to plan staffing)
#   Why order: Monday spikes, weekends drop — the pattern only exists in time
#
# Problem 1:
#   Sequence : ?
#   Target   : ?
#   Why order: ?
#
# Problem 2:
#   Sequence : ?
#   Target   : ?
#   Why order: ?
#
# Problem 3:
#   Sequence : ?
#   Target   : ?
#   Why order: ?
#
# See the Answer Key at the end of this chapter.
# ─────────────────────────────────────────────────────────────────────────────

print('Exercise 6.1 complete.')

---
# 6.2 Vanilla RNN

## The reading-a-sentence analogy

A vanilla RNN works like a reader who keeps a very short notepad.

At each word (time step), the reader:
1. Looks at the current word
2. Glances at their current notepad summary
3. Erases the notepad and writes a new summary combining both

The problem: the notepad is small and gets **completely rewritten at every step**. By the time the reader reaches word 50, almost nothing from word 1 survives. The early context is washed out.

## The maths — one equation

The vanilla RNN update is a single equation:

$$h_t = \tanh(W_h \cdot h_{t-1} + W_x \cdot x_t + b)$$

In plain English: take the previous hidden state and the current input, mix them through two learned weight matrices, add a bias, and squash the result through $\tanh$ to keep values in the range (−1, 1).

That is the entire recurrent computation. Everything that makes RNNs powerful — and everything that limits them — follows from this one line.

**Table 6.1 — The parts of an RNN**

| Symbol | Name | What it is |
|--------|------|------------|
| $x_t$ | Input at step $t$ | One element of the sequence (one week's CO₂ reading, one word token) |
| $h_{t-1}$ | Previous hidden state | The network's memory of everything before step $t$ |
| $h_t$ | New hidden state | Updated memory after reading $x_t$ |
| $W_x$ | Input weight matrix | How much to weight the current input |
| $W_h$ | Recurrent weight matrix | How much to weight the previous memory |
| $\tanh$ | Activation | Keeps hidden state values bounded between −1 and 1 |

## The vanishing gradient problem — explained without maths

Here is the key weakness of the vanilla RNN, explained as a game of telephone.

Imagine 50 people standing in a line. A message is whispered from person 1 to person 2, then person 2 passes it to person 3, and so on. Each person slightly garbles the message before passing it on.

By the time the message reaches person 50, the original message from person 1 is almost unrecognisable.

This is exactly what happens during backpropagation in a vanilla RNN. When the network learns, it sends error signals *backwards* through the time steps — from step 50 back to step 1. But at each step, the gradient is multiplied by the recurrent weight matrix $W_h$. If the values in $W_h$ are slightly less than 1, multiplying 50 times produces a number close to zero. The gradient **vanishes** before it reaches the early steps.

The result: **the network cannot learn from context that happened more than ~10–20 steps ago**. On short sequences this is fine. On longer ones — a year of weekly data, a multi-sentence document — the early signal is lost.

> **Why does this matter for forecasting?**  
> CO₂ has a yearly cycle. To forecast next week's reading, the model needs to remember what happened 52 weeks ago. A vanilla RNN simply cannot hold onto information that long. Section 6.3 shows how LSTMs solve this.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2a — Vanilla RNN model
#
# nn.RNN(batch_first=True) expects input shape: (batch, seq_len, input_size)
# It returns:
#   output : (batch, seq_len, hidden_size) — hidden state at every step
#   h_n    : (num_layers, batch, hidden_size) — hidden state at the LAST step
#
# For sequence-to-one forecasting we only need the final hidden state h_n.
# ─────────────────────────────────────────────────────────────────────────────

class VanillaRNN(nn.Module):
    """Single-layer vanilla RNN — predicts one value from a sequence."""
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity='tanh',
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)      # h_n: (num_layers, batch, hidden_size)
        return self.fc(h_n[-1])   # use last layer's final hidden state


rnn_demo = VanillaRNN(input_size=1, hidden_size=32)
n_params = sum(p.numel() for p in rnn_demo.parameters())
print('VanillaRNN architecture:')
print(rnn_demo)
print(f'\nTrainable parameters: {n_params:,}')
print()

# Quick shape check — does it run?
x_demo = torch.randn(4, 10, 1)   # batch=4, seq_len=10, features=1
y_demo = rnn_demo(x_demo)
print(f'Input shape : {tuple(x_demo.shape)}')
print(f'Output shape: {tuple(y_demo.shape)}  (one prediction per sample — correct)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2b — Demonstrate the vanishing gradient problem
#
# We train a vanilla RNN on two sine-wave datasets:
#   Short sequence (seq_len=10)  — RNN succeeds
#   Long  sequence (seq_len=60)  — RNN struggles; gradient vanishes
#
# Figure 6.2 — RNN loss curves: short vs long sequences
# ─────────────────────────────────────────────────────────────────────────────

def make_sine_dataset(seq_len, n_samples=2000):
    """Return (X, y) tensors from a noisy sine wave.
    X shape: (n_samples, seq_len, 1)   y shape: (n_samples, 1)
    """
    np.random.seed(SEED)
    t = np.linspace(0, 8 * np.pi, n_samples + seq_len + 1)
    s = np.sin(t) + 0.02 * np.random.randn(len(t))
    X = np.stack([s[i : i + seq_len] for i in range(n_samples)])
    y = s[seq_len : seq_len + n_samples].reshape(-1, 1)
    return (
        torch.tensor(X, dtype=torch.float32).unsqueeze(-1),
        torch.tensor(y, dtype=torch.float32),
    )

def make_loaders(X, y, val_split=0.15, batch_size=64):
    n = len(X); split = int(n * (1 - val_split))
    from torch.utils.data import TensorDataset, DataLoader
    tr = DataLoader(TensorDataset(X[:split], y[:split]), batch_size=batch_size, shuffle=True)
    va = DataLoader(TensorDataset(X[split:], y[split:]), batch_size=batch_size, shuffle=False)
    return tr, va

SHORT, LONG = 10, 60
crit_rnn = nn.MSELoss()
rnn_results = {}

for seq_len, label in [(SHORT, f'RNN  seq={SHORT}'), (LONG, f'RNN  seq={LONG}')]:
    X, y = make_sine_dataset(seq_len)
    tr, va = make_loaders(X, y)
    torch.manual_seed(SEED)
    m = VanillaRNN(hidden_size=32).to(device)
    _, val_h = run_training(m, tr, va, crit_rnn,
                            optim.Adam(m.parameters(), lr=1e-3),
                            num_epochs=40, device=device, verbose_every=None)
    rnn_results[label] = val_h
    print(f'{label}  →  final val MSE: {val_h[-1]:.4f}')

# ── Figure 6.2 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')
colors = ['#27ae60', '#e74c3c']
for (label, vals), color in zip(rnn_results.items(), colors):
    ax.plot(vals, color=color, lw=2.0, label=label)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE')
ax.set_title('Figure 6.2 — Vanilla RNN: short sequences work, long sequences plateau',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend()
plt.tight_layout(); plt.show()
print()
print(f'The long-sequence RNN plateaus near loss={rnn_results[f"RNN  seq={LONG}"][-1]:.4f}.')
print('Gradient vanishes before it can reach the early steps. Section 6.3 fixes this.')

### 📝 Exercise 6.2 — Observe the Vanishing Gradient

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.2
#
# Task: Train a VanillaRNN on the LONG sequence dataset (seq_len=60)
#       but with a much larger hidden size (hidden_size=128).
#
#       Does increasing the model's capacity fix the vanishing gradient?
#       Print the final validation loss and compare it to the hidden=32 result.
#
# After running, answer these questions as comments:
#   Q1. Did hidden_size=128 help meaningfully?
#   Q2. Is the vanishing gradient a problem of model size, or model design?
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
# rnn_big = VanillaRNN(hidden_size=128).to(device)
# X_long, y_long = make_sine_dataset(60)
# tr_long, va_long = make_loaders(X_long, y_long)
# _, big_val = run_training(rnn_big, tr_long, va_long, crit_rnn,
#                           optim.Adam(rnn_big.parameters(), lr=1e-3),
#                           num_epochs=40, device=device, verbose_every=10)
# print(f'hidden=32  final val: {rnn_results["RNN  seq=60"][-1]:.4f}')
# print(f'hidden=128 final val: {big_val[-1]:.4f}')
#
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.2 complete.')

---
# 6.3 LSTM

## The core idea: two memory channels

The **Long Short-Term Memory (LSTM)** network, introduced by Hochreiter and Schmidhuber in 1997, solves the vanishing gradient problem with one elegant idea: instead of a single hidden state that gets completely overwritten at every step, the LSTM keeps **two separate memory channels**.

| Channel | Name | Role |
|---------|------|------|
| $h_t$ | **Hidden state** | Short-term output — passed to the next step and to the prediction layer |
| $c_t$ | **Cell state** | Long-term memory — travels through the sequence with only small, controlled changes |

Think of it as the difference between your working memory (what you are actively thinking about right now) and your long-term memory (knowledge you have accumulated over years). The hidden state is working memory; the cell state is long-term memory.

## The three gates — selective memory

The cell state is not just a second hidden state. It is protected by **three learnable gates** that control what information flows in, out, or gets erased. Each gate is a sigmoid function that outputs a value between 0 and 1 — acting as a smooth on/off switch.

```
  ┌──────────────────────────────────────────────────────┐
  │                   LSTM Cell                          │
  │                                                      │
  │  c_{t-1} ──[FORGET]──────────────[ADD]──── c_t      │
  │                │                   │                 │
  │             Erase old          Write new             │
  │             memories           memories              │
  │                │                   │                 │
  │            [INPUT gate]        [INPUT gate]          │
  │                                                      │
  │  h_{t-1}, x_t ─────────────── [OUTPUT gate] ── h_t  │
  └──────────────────────────────────────────────────────┘
```

*Figure 6.3 — Inside an LSTM cell. The cell state $c_t$ flows across the top with only additive updates. The three gates control what is forgotten, written, and read.*

**Forget gate** — decides what to erase from long-term memory:  
*"Last year's promotion is no longer relevant — forget the uplift pattern."*

**Input gate** — decides what new information to write:  
*"This week's unusual spike is important — remember it."*

**Output gate** — decides what to surface from memory right now:  
*"The seasonal cycle is relevant here — use it for this prediction."*

## Why this solves the vanishing gradient

The vanilla RNN multiplies the hidden state by $W_h$ at every step. Multiply by a number slightly less than 1, fifty times, and you get essentially zero.

The cell state is updated **additively**: $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$. Addition preserves gradient magnitude — it does not shrink it. When the forget gate is near 1 ("keep this memory"), gradients flow backward through the cell state across many steps without shrinking. The network can genuinely learn from context that is 52 steps (one year) or 200 steps ago.

## RNN vs LSTM — a direct comparison

| Feature | Vanilla RNN | LSTM |
|---------|-------------|------|
| Memory channels | 1 (hidden state only) | 2 (hidden state + cell state) |
| Update mechanism | Overwrite (multiplicative) | Additive with gating |
| Long-range dependencies | Struggles beyond ~20 steps | Handles hundreds of steps |
| Parameters (same hidden size) | ~1× | ~4× (one set per gate) |
| Training stability | Can explode or vanish | Much more stable |
| PyTorch class | `nn.RNN` | `nn.LSTM` |
| When to use | Short sequences (<20 steps) | Sequences of any length |

> **Rule of thumb:** If your sequence is longer than 20 steps, use an LSTM. The extra parameters are worth it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3a — LSTM model
#
# nn.LSTM is a drop-in replacement for nn.RNN.
# It returns (output, (h_n, c_n)) — note the tuple.
# We unpack (h_n, c_n) and use only h_n for the prediction head.
# ─────────────────────────────────────────────────────────────────────────────

class LSTMNet(nn.Module):
    """LSTM for sequence-to-one regression or classification.
    Constructor kwargs match model_config for ModelPipeline compatibility.
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 dropout=0.2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)      # h_n: (num_layers, batch, hidden_size)
        out = self.dropout(h_n[-1])     # last layer, final time step
        return self.fc(out)             # (batch, output_size)


lstm_demo = LSTMNet(input_size=1, hidden_size=64, num_layers=2, dropout=0.2)
n_rnn  = sum(p.numel() for p in VanillaRNN(hidden_size=64).parameters())
n_lstm = sum(p.numel() for p in lstm_demo.parameters())
print('LSTMNet architecture:')
print(lstm_demo)
print(f'\nVanillaRNN parameters (hidden=64): {n_rnn:,}')
print(f'LSTMNet   parameters (hidden=64): {n_lstm:,}')
print(f'LSTM has {n_lstm / n_rnn:.1f}× more parameters — 4 weight sets, one per gate.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3b — Compare LSTM vs RNN on the long sequence problem
#
# Figure 6.4 — LSTM vs vanilla RNN on seq_len=60
# ─────────────────────────────────────────────────────────────────────────────

X_long, y_long = make_sine_dataset(LONG)
tr_long, va_long = make_loaders(X_long, y_long)

torch.manual_seed(SEED)
lstm_seq = LSTMNet(input_size=1, hidden_size=64, num_layers=2, dropout=0.2).to(device)
print('Training LSTM on long sequences (seq_len=60):')
_, lstm_long_val = run_training(
    lstm_seq, tr_long, va_long, nn.MSELoss(),
    optim.Adam(lstm_seq.parameters(), lr=1e-3),
    num_epochs=40, device=device, verbose_every=10,
)

# ── Figure 6.4 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')
ep = range(1, 41)
ax.plot(ep, rnn_results[f'RNN  seq={LONG}'], color='#e74c3c', lw=2.0, label='Vanilla RNN (seq=60)')
ax.plot(ep, lstm_long_val,                   color='#2980b9', lw=2.5, label='LSTM        (seq=60)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE')
ax.set_title('Figure 6.4 — LSTM solves what the vanilla RNN cannot',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend()
plt.tight_layout(); plt.show()
print(f'\nFinal val MSE — RNN : {rnn_results[f"RNN  seq={LONG}"][-1]:.4f}')
print(f'Final val MSE — LSTM: {lstm_long_val[-1]:.4f}')
print('The LSTM converges clearly; the RNN plateaus near its starting loss.')

### 📝 Exercise 6.3 — Replace RNN with LSTM

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.3
#
# Task: Train an LSTMNet on the SHORT sequence dataset (seq_len=10).
#       Plot its val loss alongside the RNN short-sequence result.
#
# After running, answer these questions as comments:
#   Q1. On short sequences, is the LSTM clearly better than the RNN?
#       What does this tell you about when to use each architecture?
#   Q2. The LSTM has ~4× more parameters than the RNN for the same hidden size.
#       Is that extra cost always justified?
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
# X_short, y_short = make_sine_dataset(SHORT)
# tr_short, va_short = make_loaders(X_short, y_short)
# lstm_short = LSTMNet(input_size=1, hidden_size=32, num_layers=1, dropout=0.0).to(device)
# _, lstm_short_val = run_training(
#     lstm_short, tr_short, va_short, nn.MSELoss(),
#     optim.Adam(lstm_short.parameters(), lr=1e-3),
#     num_epochs=40, device=device, verbose_every=None)
#
# fig, ax = plt.subplots(figsize=(9, 4))
# ax.plot(rnn_results[f'RNN  seq={SHORT}'], color='#27ae60', lw=2.0, label=f'RNN  seq={SHORT}')
# ax.plot(lstm_short_val,                   color='#2980b9', lw=2.0, label=f'LSTM seq={SHORT}')
# ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE'); ax.legend()
# ax.set_title('Exercise 6.3 — RNN vs LSTM on short sequences')
# plt.tight_layout(); plt.show()
#
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.3 complete.')

---
# 6.4 Business Application: CO₂ Forecasting

## The business problem

Atmospheric CO₂ concentration is one of the most consequential time series in the world. Measured weekly at the Mauna Loa Observatory in Hawaii since 1958, it captures humanity's carbon footprint in a single number. Every organisation with a climate risk strategy, carbon credit programme, or net-zero commitment needs accurate CO₂ forecasts.

It is also a perfect teaching dataset, because its structure is unusually clean and highly learnable:

- **A strong upward trend** — concentration has risen from 316 ppm in 1958 to 372 ppm by 2001, driven by fossil fuel emissions (~0.9 ppm per year on average)
- **A precise yearly cycle** — CO₂ dips each northern-hemisphere summer as vegetation absorbs it, then rises each winter as plant activity falls (the "Keeling Curve" sawtooth)

A well-trained LSTM should learn both patterns and produce visibly accurate forecasts. If your model's predictions track the seasonal sawtooth into the future, you have built something real.

## The dataset

The data is built directly into `statsmodels` — one function call, no download, no account.

| Property | Value |
|----------|-------|
| Source | Scripps Institution of Oceanography / NOAA |
| Frequency | Weekly |
| Date range | 30 March 1958 – 30 December 2001 |
| Total observations | 2,284 weekly readings |
| Target variable | CO₂ concentration (parts per million, ppm) |
| Licence | Public domain |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4a — Load Mauna Loa CO₂ data
#
# sm.datasets.co2 ships with statsmodels. No file download required.
# resample('W') makes the index strictly weekly; ffill() fills 2 missing weeks.
# ─────────────────────────────────────────────────────────────────────────────

co2_raw = sm.datasets.co2.load_pandas().data
co2_raw = co2_raw.resample('W').mean().ffill()

daily = (
    co2_raw
    .reset_index()
    .rename(columns={'index': 'date', 'co2': 'co2_ppm'})
)

print(f'Loaded {len(daily):,} weekly CO₂ readings')
print(f'Date range : {daily.date.min().date()}  →  {daily.date.max().date()}')
print(f'CO₂ (ppm)  : min={daily.co2_ppm.min():.1f}  max={daily.co2_ppm.max():.1f}  '
      f'mean={daily.co2_ppm.mean():.1f}')
print()
print(daily.head())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4b — Exploratory data analysis
#
# Figure 6.5 — Mauna Loa CO₂: the full series and the seasonal cycle
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.patch.set_facecolor('white')

# Panel (a): full 43-year series
axes[0].set_facecolor('white')
axes[0].plot(daily.date, daily.co2_ppm, color='#2980b9', lw=0.9, alpha=0.9)
axes[0].set_title('(a) Mauna Loa CO₂ — Full History (1958–2001)',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[0].set_ylabel('CO₂ (ppm)', fontsize=10)
axes[0].annotate('Upward trend\n+0.9 ppm/year', xy=(daily.date.iloc[1000], 340),
                 fontsize=9, color='#1F3864',
                 arrowprops=dict(arrowstyle='->', color='#1F3864'),
                 xytext=(daily.date.iloc[600], 356))

# Panel (b): 5-year zoom to show the sawtooth seasonal cycle
zoom = daily[daily.date.dt.year.between(1990, 1994)]
axes[1].set_facecolor('white')
axes[1].plot(zoom.date, zoom.co2_ppm, color='#e74c3c', lw=2.0)
axes[1].set_title('(b) 1990–1994 zoom — the yearly sawtooth cycle '
                  '(summer dip as plants absorb CO₂, winter rise as they stop)',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[1].set_ylabel('CO₂ (ppm)', fontsize=10)

# Mark a summer minimum and winter maximum
summer_idx = zoom.co2_ppm.idxmin()
winter_idx = zoom.co2_ppm.idxmax()
axes[1].scatter(zoom.loc[summer_idx, 'date'], zoom.loc[summer_idx, 'co2_ppm'],
               color='#27ae60', zorder=5, s=80, label='Summer minimum')
axes[1].scatter(zoom.loc[winter_idx, 'date'], zoom.loc[winter_idx, 'co2_ppm'],
               color='#e74c3c', zorder=5, s=80, label='Winter maximum')
axes[1].legend(fontsize=9)

fig.suptitle('Figure 6.5 — Mauna Loa CO₂: strong trend + precise yearly seasonality',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4c — Preprocessing: scale and build sliding windows
#
# The LSTM receives one sliding window at a time:
#   Input  : last WINDOW_SIZE weeks of CO₂ readings → shape (WINDOW_SIZE, 1)
#   Target : the next week's reading                → shape (1,)
#
# WINDOW_SIZE = 52 means "use the past year to predict next week."
# This is long enough to capture the full yearly seasonal cycle.
#
# We use a TEMPORAL split — the last 20% of time is the validation set.
# A random split would leak future values into training (data leakage).
# ─────────────────────────────────────────────────────────────────────────────

WINDOW_SIZE = 52   # 1 year of weekly look-back
VAL_FRAC    = 0.20

# 1. Scale to [0, 1] using only the training portion
values   = daily.co2_ppm.values.reshape(-1, 1)
n        = len(values)
split    = int(n * (1 - VAL_FRAC))

scaler = MinMaxScaler()
scaler.fit(values[:split])               # fit on train only — no leakage
scaled = scaler.transform(values).flatten()

# 2. Slide a window across the series
def make_windows(series, window_size):
    """Return (X, y) tensors.
    X: (n_windows, window_size, 1)  —  the input sequence
    y: (n_windows, 1)               —  the next value to predict
    """
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size])
    X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)  # (N, W, 1)
    y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(-1)  # (N, 1)
    return X, y

X_all, y_all = make_windows(scaled, WINDOW_SIZE)

# Temporal split
tr_split = split - WINDOW_SIZE
X_tr, y_tr = X_all[:tr_split], y_all[:tr_split]
X_va, y_va = X_all[tr_split:], y_all[tr_split:]

co2_train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
co2_val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=32, shuffle=False)

print(f'Window size    : {WINDOW_SIZE} weeks (1 year of look-back)')
print(f'Training windows : {len(X_tr):,}  |  shape: {tuple(X_tr.shape)}')
print(f'Validation windows: {len(X_va):,}  |  shape: {tuple(X_va.shape)}')
print()
print('Each window looks like this:')
print(f'  Input  : 52 consecutive weekly CO₂ readings (one full year)')
print(f'  Target : the reading in week 53')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4d — Train the CO₂ forecast LSTM
# ─────────────────────────────────────────────────────────────────────────────

CO2_CONFIG = dict(
    input_size  = 1,
    hidden_size = 64,
    num_layers  = 2,
    dropout     = 0.2,
    output_size = 1,
)

torch.manual_seed(SEED)
co2_model = LSTMNet(**CO2_CONFIG).to(device)

co2_crit  = nn.MSELoss()
co2_opt   = optim.Adam(co2_model.parameters(), lr=1e-3)
co2_sched = optim.lr_scheduler.CosineAnnealingLR(co2_opt, T_max=60)
co2_es    = EarlyStopping(patience=10)

print(f'Model parameters: {sum(p.numel() for p in co2_model.parameters()):,}')
print('Training CO₂ forecast LSTM (up to 60 epochs, CosineAnnealingLR):')
print()

co2_train_hist, co2_val_hist = run_training(
    co2_model, co2_train_loader, co2_val_loader, co2_crit, co2_opt,
    num_epochs=60, device=device,
    scheduler=co2_sched, early_stopping=co2_es, verbose_every=5,
)
print('\nTraining complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4e — Evaluate: MAE, MAPE, and forecast plot
#
# MAE  = mean absolute error in ppm (how far off on average, in real units)
# MAPE = mean absolute percentage error (scale-independent, easy to interpret)
#
# Figure 6.6 — Forecast vs actual, with loss curves
# ─────────────────────────────────────────────────────────────────────────────

co2_model.eval()
preds_sc, actuals_sc = [], []
with torch.no_grad():
    for X_b, y_b in co2_val_loader:
        preds_sc.append(co2_model(X_b.to(device)).cpu().numpy())
        actuals_sc.append(y_b.numpy())

preds_sc   = np.concatenate(preds_sc).flatten()
actuals_sc = np.concatenate(actuals_sc).flatten()

# Inverse-transform back to ppm
preds   = scaler.inverse_transform(preds_sc.reshape(-1,   1)).flatten()
actuals = scaler.inverse_transform(actuals_sc.reshape(-1, 1)).flatten()

mae  = np.mean(np.abs(preds - actuals))
mape = np.mean(np.abs((preds - actuals) / actuals)) * 100

print(f'Validation MAE  : {mae:.3f} ppm')
print(f'Validation MAPE : {mape:.2f}%')
print()
print('Interpretation:')
print(f'  The model is off by {mae:.2f} ppm on average — that is about {mae/3.5*100:.0f}% of'
      f' one year\'s typical increase ({3.5:.1f} ppm/year).')

# ── Dates for the validation period ──────────────────────────────────────────
val_dates = daily.date.values[split + WINDOW_SIZE : split + WINDOW_SIZE + len(actuals)]

# ── Figure 6.6 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.patch.set_facecolor('white')

# Panel (a): forecast vs actual
axes[0].set_facecolor('white')
axes[0].plot(val_dates, actuals, color='#1F3864', lw=1.5, label='Actual CO₂')
axes[0].plot(val_dates, preds,   color='#e74c3c', lw=1.5, label='LSTM forecast', alpha=0.85)
axes[0].set_title(f'(a) CO₂ Forecast vs Actual — Validation Period  '
                  f'(MAE: {mae:.2f} ppm  MAPE: {mape:.1f}%)',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[0].set_ylabel('CO₂ (ppm)', fontsize=10)
axes[0].legend(fontsize=9)

# Panel (b): loss curves
axes[1].set_facecolor('white')
ep = range(1, len(co2_train_hist) + 1)
axes[1].plot(ep, co2_train_hist, color='#2980b9', lw=2.0, label='Train MSE')
axes[1].plot(ep, co2_val_hist,   color='#e67e22', lw=2.0, label='Val MSE',  linestyle='--')
axes[1].set_title('(b) Training Loss Curves', fontsize=11, fontweight='bold', color='#1F3864')
axes[1].set_xlabel('Epoch', fontsize=10)
axes[1].set_ylabel('MSE Loss (scaled)', fontsize=10)
axes[1].legend(fontsize=9)

fig.suptitle('Figure 6.6 — CO₂ Forecast LSTM: the model has learned both trend and seasonality',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

### 📝 Exercise 6.4 — End-to-End CO₂ Forecasting

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.4
#
# Task A: Change WINDOW_SIZE to 26 (half a year).
#         Retrain the same LSTM. Does MAE improve or worsen?
#         Think about why: what seasonal context does a 26-week window miss?
#
# Task B: Change hidden_size to 32 (smaller model).
#         How does MAE change? Is the smaller model still acceptable?
#
# Task C: Answer as comments:
#   Q1. We used a temporal split (last 20% = validation).
#       Why would a random split be wrong for time series?
#   Q2. MAPE divides by the actual value. Why is it useful for CO₂ forecasting
#       even though CO₂ values are never near zero?
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# WINDOW_SIZE_A = 26
# X_a, y_a = make_windows(scaled, WINDOW_SIZE_A)
# ...  (same train/val split and training loop)

# Task B
# CO2_CONFIG_B = {**CO2_CONFIG, 'hidden_size': 32}
# ...

# Task C
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.4 complete.')

---
# 6.5 Pipeline: Saving and Deploying the Forecast Model

Every chapter from Chapter 3 onward closes with the same professional habit: wrap the trained model in a `ModelPipeline` and save it. For sequence models, there is one important addition — the **window configuration** (how many past steps the model expects) must be saved alongside the weights.

If you load a saved LSTM and feed it 30 steps when it was trained on 52, you will get silent wrong predictions. Saving the window size in `model_config` and checking it in `validate_input()` catches this mistake immediately.

**Table 6.2 — What the pipeline saves for the CO₂ forecast model**

| What is saved | Why it matters |
|--------------|----------------|
| Model architecture config | Rebuilds the exact same network on load |
| `window_size` (in model_config) | `validate_input()` checks incoming sequence length |
| Model weights (`state_dict`) | The learned parameters |
| Fitted `MinMaxScaler` | New data must be scaled the same way as training data |
| Training history | Loss curves — useful for diagnosing and continuing training |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ModelPipeline — Chapter 6 version
#
# Identical to Chapters 3–5 with one extension:
#   validate_input() checks that incoming sequences have the correct shape
#   (batch, window_size, input_size), reading window_size from model_config.
# ─────────────────────────────────────────────────────────────────────────────

import dill

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Chapter 6 extension: validate_input checks sequence shape.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model            = model
        self.model_config     = model_config
        self.preprocessor     = preprocessor
        self.feature_names    = feature_names or []
        self.feature_ranges   = feature_ranges or {}
        self._model_class     = model_class
        self.model_class      = model_class.__name__ if model_class else type(model).__name__
        self.training_history = {'train_loss': [], 'val_loss': []}

    def save(self, path):
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved → {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    @classmethod
    def load(cls, path, device=None):
        if device is None: device = torch.device('cpu')
        ckpt       = torch.load(path, map_location=device)
        ModelClass = dill.loads(ckpt['model_class_bytes'])
        arch_keys  = {'input_size', 'hidden_size', 'num_layers', 'dropout', 'output_size'}
        arch_cfg   = {k: v for k, v in ckpt['model_config'].items() if k in arch_keys}
        model      = ModelClass(**arch_cfg)
        model.load_state_dict(ckpt['state_dict'])
        model.to(device).eval()
        pipeline                  = cls(model=model, model_config=ckpt['model_config'],
                                        preprocessor=ckpt['preprocessor'],
                                        feature_names=ckpt['feature_names'],
                                        feature_ranges=ckpt['feature_ranges'])
        pipeline._model_class     = ModelClass
        pipeline.model_class      = ckpt['model_class']
        pipeline.training_history = ckpt['training_history']
        print(f'Pipeline loaded from {path}')
        print(f'  model_class : {pipeline.model_class}')
        print(f'  window_size : {pipeline.model_config.get("window_size", "not set")}')
        return pipeline

    def validate_input(self, X):
        """Check that X has shape (batch, window_size, input_size)."""
        expected_w = self.model_config.get('window_size')
        expected_f = self.model_config.get('input_size', 1)
        if not isinstance(X, torch.Tensor):
            raise ValueError('Input must be a torch.Tensor.')
        if X.ndim != 3:
            raise ValueError(f'Expected 3-D tensor (batch, window, features), got {X.shape}.')
        if expected_w is not None and X.shape[1] != expected_w:
            raise ValueError(f'Window size mismatch: model expects {expected_w}, got {X.shape[1]}.')
        if X.shape[2] != expected_f:
            raise ValueError(f'Feature mismatch: model expects {expected_f}, got {X.shape[2]}.')
        return True

    def predict(self, X_raw_np):
        """X_raw_np: numpy array (batch, window_size) — raw unscaled CO₂ values.
        Returns predictions in original CO₂ units (ppm).
        """
        if X_raw_np.ndim == 2:
            X_raw_np = X_raw_np[:, :, np.newaxis]
        flat    = X_raw_np.reshape(-1, 1)
        sc_flat = self.preprocessor.transform(flat)
        X_t     = torch.tensor(sc_flat.reshape(X_raw_np.shape), dtype=torch.float32)
        self.validate_input(X_t)
        self.model.eval()
        with torch.no_grad():
            pred_sc = self.model(X_t).cpu().numpy()
        return self.preprocessor.inverse_transform(pred_sc.reshape(-1, 1)).flatten()

    def retrain(self, train_loader, val_loader, criterion, optimiser, num_epochs, device):
        """Continue training; append results to training_history."""
        self.model.train()
        new_tr, new_va = run_training(
            self.model, train_loader, val_loader, criterion, optimiser,
            num_epochs=num_epochs, device=device, verbose_every=5,
        )
        self.training_history['train_loss'].extend(new_tr)
        self.training_history['val_loss'].extend(new_va)
        print(f'Retrain complete. Total epochs in history: {len(self.training_history["val_loss"])}')


print('ModelPipeline defined.')
print('Methods: save | load | validate_input | predict | retrain')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5a — Wrap the trained model and save
# ─────────────────────────────────────────────────────────────────────────────

co2_model_config = {
    **CO2_CONFIG,
    'window_size': WINDOW_SIZE,   # saved so validate_input() can check
}

pipeline = ModelPipeline(
    model        = co2_model,
    model_config = co2_model_config,
    preprocessor = scaler,          # fitted MinMaxScaler
    feature_names= ['co2_ppm'],
    model_class  = LSTMNet,
)
pipeline.training_history = {'train_loss': co2_train_hist, 'val_loss': co2_val_hist}

pipeline.save('co2_forecast_v1.pth')
print(f'\nWindow size stored in model_config: {pipeline.model_config["window_size"]} weeks')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5b — Load in a clean context and forecast the next 52 weeks (1 year)
#
# Autoregressive (rolling) forecast:
#   1. Seed the model with the last 52 weeks of actual data
#   2. Predict week 53
#   3. Append the prediction to the window, drop the oldest reading
#   4. Repeat for each future week
#
# Figure 6.7 — 52-week CO₂ forecast from the loaded pipeline
# ─────────────────────────────────────────────────────────────────────────────

del pipeline   # simulate a fresh session

loaded = ModelPipeline.load('co2_forecast_v1.pth')
print(f'\nTraining history: {len(loaded.training_history["val_loss"])} epochs')

HORIZON = 52   # forecast 1 year ahead

# Seed: last WINDOW_SIZE weeks of actual CO₂ values (raw ppm)
seed = daily.co2_ppm.values[-WINDOW_SIZE:].copy()
forecast = []

for _ in range(HORIZON):
    pred = loaded.predict(seed[np.newaxis, :])   # predict one step ahead
    forecast.append(pred[0])
    seed = np.append(seed[1:], pred[0])          # roll window forward

forecast      = np.array(forecast)
last_date     = daily.date.iloc[-1]
future_dates  = pd.date_range(start=last_date + pd.Timedelta(weeks=1),
                               periods=HORIZON, freq='W')

# ── Figure 6.7 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')

context = daily.tail(3 * 52)   # show last 3 years as context
ax.plot(context.date, context.co2_ppm,
        color='#1F3864', lw=1.5, label='Actual (last 3 years)')
ax.plot(future_dates, forecast,
        color='#e74c3c', lw=2.0, linestyle='--', label='52-week forecast')
ax.axvline(x=last_date, color='#aaaaaa', lw=1.2, linestyle=':')
ax.text(last_date, ax.get_ylim()[0] + 0.5, ' Forecast →', fontsize=9, color='#555')
ax.set_ylabel('CO₂ (ppm)', fontsize=10)
ax.set_title('Figure 6.7 — 52-week CO₂ Autoregressive Forecast (loaded from .pth)',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'\nForecast summary (next {HORIZON} weeks):')
print(f'  Start : {forecast[0]:.2f} ppm')
print(f'  End   : {forecast[-1]:.2f} ppm')
print(f'  Range : {forecast.min():.2f} – {forecast.max():.2f} ppm')
print()
print('The forecast should show the familiar sawtooth — seasonal dip in summer,')
print('followed by a winter rise, sitting slightly above the last actual reading.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5c — Retrain with new data and save v2
#
# In production: new weekly CO₂ readings arrive and the model should
# be updated. Here we simulate by running 10 more epochs on the val set.
# ─────────────────────────────────────────────────────────────────────────────

retrain_opt = optim.Adam(loaded.model.parameters(), lr=2e-4)

print('Retraining on new data (10 additional epochs):')
loaded.retrain(
    train_loader = co2_val_loader,   # simulate new data
    val_loader   = co2_val_loader,
    criterion    = nn.MSELoss(),
    optimiser    = retrain_opt,
    num_epochs   = 10,
    device       = device,
)

loaded.save('co2_forecast_v2.pth')
print(f'\nTotal epochs in history: {len(loaded.training_history["val_loss"])}')
print('v1 is still on disk — roll back by loading co2_forecast_v1.pth if needed.')

### 📝 Exercise 6.5 — Pipeline for Sequence Models

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.5
#
# Task A: Load co2_forecast_v1.pth and intentionally trigger a ValueError
#         by calling validate_input() with the wrong sequence length.
#         Confirm the error message is clear and informative.
#
# Task B: Load co2_forecast_v2.pth and generate a 52-week forecast.
#         Plot v1 and v2 forecasts on the same axes.
#         Do the two forecasts differ? Describe what changed.
#
# Task C: Answer as comments:
#   Q1. Why is it important to save the window_size inside model_config
#       rather than just as a separate variable in your script?
#   Q2. The forecast gets less reliable further into the future.
#       In one sentence, explain why (hint: think about what the model
#       is using as input after the first few predictions).
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# v1 = ModelPipeline.load('co2_forecast_v1.pth')
# bad_input = torch.zeros(1, 30, 1)   # wrong: model expects window_size=52
# try:
#     v1.validate_input(bad_input)
# except ValueError as e:
#     print(f'Caught expected error: {e}')

# Task B
# v2 = ModelPipeline.load('co2_forecast_v2.pth')
# ... (generate forecast, plot both)

# Task C
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.5 complete.')

---
# 6.6 ★ Bonus: Autoencoders

## What is an autoencoder?

Every model in this book so far has been **supervised**: it learns a mapping from inputs to labels. An autoencoder flips this around. It has no labels at all — its target is its own input.

The goal is to compress the input into a small **bottleneck** representation and then reconstruct the original from that compressed form. The network learns by minimising the difference between input and reconstruction.

```
Input x          Bottleneck z          Reconstruction x̂
  (20 features)    (4 features)           (20 features)

  ┌─────────┐       ┌─────┐              ┌─────────┐
  │  x₁     │       │     │              │  x̂₁    │
  │  x₂     │──────►│  z  │─────────────►│  x̂₂    │
  │  ...    │       │     │              │  ...    │
  │  x₂₀   │       └─────┘              │  x̂₂₀   │
  └─────────┘                           └─────────┘
    ENCODER          (compress)          DECODER

  Loss = MSE(x, x̂)   ← no labels needed
```

*Figure 6.8 — Autoencoder structure. The bottleneck forces the encoder to keep only the most essential information.*

## Why the bottleneck matters

Because the bottleneck has fewer dimensions than the input, the encoder is forced to discard anything that is not essential and keep only the most important structure. This is the same idea as compressing a photo into a smaller file — the compression removes redundancy while preserving the key content.

## Business applications

| Application | How the autoencoder is used |
|-------------|-----------------------------|
| **Anomaly detection** | Train on normal data. Anomalous inputs reconstruct poorly — high reconstruction error flags them |
| **Dimensionality reduction** | Use the bottleneck as a compressed feature set for downstream models |
| **Data denoising** | Train with noisy inputs and clean targets — the network learns to remove noise |
| **Feature learning** | Pre-train on unlabelled data; fine-tune on a small labelled set |

**Anomaly detection** is the most common business use. You train only on normal transactions, normal sensor readings, or normal images. The model learns to reconstruct "normal" well. When an anomaly arrives, it does not match any pattern the encoder has learned — reconstruction error is high. Flag it.

> **Why here, in Chapter 6?** You now have the full supervised toolkit: FFNs (Ch. 4), CNNs (Ch. 5), LSTMs (this chapter). Autoencoders use the same building blocks — `nn.Linear`, `nn.Conv2d`, `nn.LSTM` — but flip the task. This is the bridge to unsupervised learning and directly prepares you for Chapter 7, where large language models learn from unlabelled text using a related self-supervised idea.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6a — FFN Autoencoder (tabular data)
#
# Use case: anomaly detection on transaction records or sensor readings.
# Train on normal data. At inference, high reconstruction error = anomaly.
#
# The only change from a regular FFN:
#   criterion = nn.MSELoss()
#   loss      = criterion(model(X_batch), X_batch)   ← target IS the input
# ─────────────────────────────────────────────────────────────────────────────

class FFNAutoencoder(nn.Module):
    """Symmetric FFN autoencoder.

    Encoder : input_dim → hidden → bottleneck_dim
    Decoder : bottleneck_dim → hidden → input_dim
    Loss    : MSELoss(x_hat, x)  — no labels needed
    """
    def __init__(self, input_dim=20, bottleneck_dim=4):
        super().__init__()
        hidden = max(input_dim // 2, bottleneck_dim + 1)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim,     hidden),          nn.ReLU(),
            nn.Linear(hidden,        bottleneck_dim),  # ← bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden),          nn.ReLU(),
            nn.Linear(hidden,         input_dim),       # ← reconstruction
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def encode(self, x):
        """Return the bottleneck representation only."""
        return self.encoder(x)


ffn_ae = FFNAutoencoder(input_dim=20, bottleneck_dim=4)
print('FFN Autoencoder:')
print(ffn_ae)
print(f'Parameters: {sum(p.numel() for p in ffn_ae.parameters()):,}')
print()

# Training recipe (same loop as throughout the book)
print('Training recipe:')
print('  criterion = nn.MSELoss()')
print('  # Note: target is the INPUT itself, not a label')
print('  loss = criterion(model(X_batch), X_batch)')
print()
print('Anomaly detection recipe:')
print('  recon_err = ((model(X) - X) ** 2).mean(dim=1)   # per-sample MSE')
print('  threshold = recon_err.mean() + 2 * recon_err.std()')
print('  anomalies = X[recon_err > threshold]')
print()

# Smoke test
x_test = torch.randn(5, 20)
x_hat  = ffn_ae(x_test)
print(f'Smoke test — input: {tuple(x_test.shape)}  reconstruction: {tuple(x_hat.shape)}  ✓')

### 📝 Exercises 6.6 — Autoencoders

Three exercises — one for each architecture type. Each exercise builds directly on what you already know.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.6a — FFN Autoencoder (beginner)
#
# Background: A bank has 10,000 normal credit card transactions.
#   Each transaction has 15 numeric features (amount, merchant category,
#   time of day, etc.). No fraud labels are available.
#
# Your task:
#   Step 1. Generate 10,000 synthetic "normal" transactions (random normal data).
#   Step 2. Train an FFNAutoencoder(input_dim=15, bottleneck_dim=5) on these.
#   Step 3. Generate 10 "anomalous" transactions (add large random noise).
#   Step 4. Compute the per-sample reconstruction error for both groups.
#   Step 5. Print the mean error for normal vs anomalous transactions.
#           The anomalous ones should have higher error.
#
# Hint: use nn.MSELoss(reduction='none') to get per-sample losses,
#       then average over the feature dimension.
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)

# Step 1 — generate data
# X_normal   = torch.randn(10_000, 15)
# X_anomalous = torch.randn(10, 15) + 5.0   # anomalies are shifted far from normal

# Step 2 — create model and train
# ae_fraud = FFNAutoencoder(input_dim=15, bottleneck_dim=5)
# ... (use nn.MSELoss and a simple training loop)

# Step 3–5 — evaluate reconstruction errors
# ...
# print(f'Mean error — normal     : ?')
# print(f'Mean error — anomalous  : ?')

# See the Answer Key at the end of this chapter.
print('Exercise 6.6a complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.6b — CNN Autoencoder (intermediate)
#
# Background: A factory wants to detect defective product images without
#   manually labelling thousands of photos. They only have unlabelled images.
#
# Your task: Define a CNN autoencoder for 28×28 greyscale images.
#
# The encoder should compress (1, 28, 28) → small feature map:
#   Conv2d(1, 16, 3, padding=1) → ReLU → MaxPool2d(2,2)   → (16, 14, 14)
#   Conv2d(16, 8, 3, padding=1) → ReLU → MaxPool2d(2,2)   → (8,  7,  7)
#
# The decoder should upsample back to (1, 28, 28):
#   ConvTranspose2d(8, 16, 2, stride=2) → ReLU             → (16, 14, 14)
#   ConvTranspose2d(16, 1, 2, stride=2) → Sigmoid          → (1,  28, 28)
#   (Sigmoid keeps output in [0,1] to match normalised image pixels)
#
# Tasks:
#   1. Define the CNNAutoencoder class with encoder and decoder as above.
#   2. Verify it runs: input (2, 1, 28, 28) → output (2, 1, 28, 28).
#   3. Print the parameter count and compare to FFNAutoencoder(input_dim=784).
#      Why does the CNN have far fewer parameters?
# ─────────────────────────────────────────────────────────────────────────────

# class CNNAutoencoder(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#             nn.Conv2d(16, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#         )
#         self.decoder = nn.Sequential(
#             nn.ConvTranspose2d(8, 16, 2, stride=2), nn.ReLU(),
#             nn.ConvTranspose2d(16, 1, 2, stride=2), nn.Sigmoid(),
#         )
#     def forward(self, x):
#         return self.decoder(self.encoder(x))
#
# cnn_ae   = CNNAutoencoder()
# x_img    = torch.randn(2, 1, 28, 28)
# x_hat    = cnn_ae(x_img)
# print(f'Input: {tuple(x_img.shape)}  Output: {tuple(x_hat.shape)}')
# print(f'CNN AE parameters: {sum(p.numel() for p in cnn_ae.parameters()):,}')
# ffn_flat = FFNAutoencoder(input_dim=784, bottleneck_dim=64)
# print(f'FFN AE parameters: {sum(p.numel() for p in ffn_flat.parameters()):,}')

# See the Answer Key at the end of this chapter.
print('Exercise 6.6b complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.6c — LSTM Autoencoder (advanced)
#
# Background: A utility company wants to detect unusual energy consumption
#   patterns without labelling historical data.
#
# Your task: Define a sequence autoencoder for time series of length 52.
#
# The encoder:
#   An nn.LSTM that reads the full sequence and produces a bottleneck
#   (the final hidden state h_n).
#
# The decoder:
#   A second nn.LSTM that receives the bottleneck (repeated seq_len times)
#   and produces a reconstructed sequence of the same length.
#   A final nn.Linear maps each step's hidden output to one value.
#
# Tasks:
#   1. Define LSTMAutoencoder(input_size=1, hidden_size=32, seq_len=52).
#   2. Verify: input (4, 52, 1) → output (4, 52, 1).
#   3. Write the training loop (one mini-batch) using MSELoss.
#      Remember: loss = criterion(x_hat, x)  — target is the input itself.
#   4. For a trained model: which windows would you flag as anomalies?
#      Write the anomaly detection logic in 3 lines.
# ─────────────────────────────────────────────────────────────────────────────

# class LSTMAutoencoder(nn.Module):
#     def __init__(self, input_size=1, hidden_size=32, seq_len=52):
#         super().__init__()
#         self.seq_len = seq_len
#         self.encoder_lstm = nn.LSTM(input_size,  hidden_size, batch_first=True)
#         self.decoder_lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
#         self.output_layer = nn.Linear(hidden_size, input_size)
#
#     def forward(self, x):
#         # x: (batch, seq_len, input_size)
#         _, (h_n, c_n) = self.encoder_lstm(x)
#         # Repeat bottleneck across all decoder steps
#         decoder_input   = h_n[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
#         decoded, _      = self.decoder_lstm(decoder_input, (h_n, c_n))
#         return self.output_layer(decoded)  # (batch, seq_len, input_size)
#
# lstm_ae = LSTMAutoencoder(input_size=1, hidden_size=32, seq_len=52)
# x_seq   = torch.randn(4, 52, 1)
# x_hat   = lstm_ae(x_seq)
# print(f'Input: {tuple(x_seq.shape)}  Output: {tuple(x_hat.shape)}')
#
# # Training (one batch):
# criterion = nn.MSELoss()
# loss = criterion(x_hat, x_seq)   # target IS the input
# print(f'Reconstruction loss (untrained): {loss.item():.4f}')
#
# # Anomaly detection:
# recon_err = ((lstm_ae(x_seq) - x_seq) ** 2).mean(dim=[1, 2])  # per sample
# threshold = recon_err.mean() + 2 * recon_err.std()
# anomalies = (recon_err > threshold).nonzero(as_tuple=True)[0]
# print(f'Anomalous samples: {anomalies.tolist()}')

# See the Answer Key at the end of this chapter.
print('Exercise 6.6c complete.')

---
## Chapter 6 Summary

| Concept | Key takeaway |
|---------|-------------|
| **Why sequences are different** | Order carries meaning — the past shapes what comes next; FFNs treat every input as independent |
| **Hidden state** | The network's running memory — updated at every step, carries context forward |
| **Parameter sharing** | The same RNN cell (same weights) is applied at every time step — no extra parameters for longer sequences |
| **Vanilla RNN** | `nn.RNN` — simple, fast, but gradients vanish through long sequences; fails beyond ~20 steps |
| **Vanishing gradient** | Like a whispered message through 50 people — the original signal is garbled by the time it reaches the end |
| **LSTM** | `nn.LSTM` — two memory channels (hidden state + cell state); additive cell updates preserve gradients |
| **Three gates** | Forget (erase), Input (write), Output (read) — learnable on/off switches that protect long-term memory |
| **Gradient clipping** | `clip_grad_norm_(max_norm=1.0)` — prevents rare but catastrophic gradient explosions in RNNs/LSTMs |
| **Sliding window** | Converts a flat series into (X, y) pairs; always use a temporal split — random splits leak the future |
| **MAE and MAPE** | MAE in original units; MAPE as a percentage — MAPE is preferred when comparing across different scales |
| **Mauna Loa CO₂** | Built into `statsmodels`; strong trend + yearly seasonality; LSTM learns both patterns visibly |
| **Autoencoder** | Unsupervised — target is the input itself; encoder compresses, decoder reconstructs; high error = anomaly |
| **ModelPipeline (sequence)** | Same contract as Ch. 3–5; stores `window_size` in `model_config`; `validate_input()` checks tensor shape |

---

## What Comes Next

Chapter 7 introduces **Large Language Models** — the architecture that has transformed every industry that works with text. Where the LSTM in this chapter learned patterns from 2,000 weekly numbers, an LLM arrives pre-trained on hundreds of billions of tokens. Rather than train a model, Chapter 7 shows how to *direct* one: through prompt engineering, structured outputs, and a full capstone project building a business Q&A bot over a corpus of earnings call transcripts.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*

---
## Answer Key — Chapter 6 Exercises

Use these suggested answers to check your thinking. For open-ended exercises (6.1) there is no single correct answer — the sample responses show the kind of reasoning to aim for.

---

### Exercise 6.1 — Sequence Problems in Your Domain

*Open-ended. Sample responses:*

1. **Daily hospital admissions**  
   Sequence: daily emergency admission counts | Target: next week's volume | Why: flu seasons, public holidays, and heatwaves create temporal patterns — today's count depends heavily on the last 14 days

2. **Customer support ticket volume**  
   Sequence: hourly ticket count | Target: next hour's volume | Why: Monday morning spikes, lunch dips, and product-launch surges are all temporal; an FFN ignoring time would miss them entirely

3. **Retail energy consumption**  
   Sequence: hourly kilowatt readings per store | Target: next day's consumption | Why: opening hours, day of week, and seasonal heating/cooling create strong time-dependent patterns

---

### Exercise 6.2 — Observe the Vanishing Gradient

**Q1.** Increasing `hidden_size` to 128 does not fix the vanishing gradient. The model has more capacity, but the fundamental problem is *architectural* — at every step, the gradient is multiplied by `W_h`. Larger weights do not prevent this shrinkage and may make exploding gradients more likely. You will observe that `hidden=128` plateaus at roughly the same loss as `hidden=32` on the long sequence.

**Q2.** The vanishing gradient is a problem of **architecture design**, not model size. No amount of capacity can compensate for a design that multiplies gradients to near-zero at each time step. The fix is a structural change — the LSTM's additive cell state update.

```python
torch.manual_seed(SEED)
rnn_big = VanillaRNN(hidden_size=128).to(device)
X_long, y_long = make_sine_dataset(60)
tr_long, va_long = make_loaders(X_long, y_long)
_, big_val = run_training(rnn_big, tr_long, va_long, nn.MSELoss(),
                          optim.Adam(rnn_big.parameters(), lr=1e-3),
                          num_epochs=40, device=device, verbose_every=None)
print(f'hidden=32  final val: {rnn_results["RNN  seq=60"][-1]:.4f}')
print(f'hidden=128 final val: {big_val[-1]:.4f}')
# Both plateau near the same value — capacity does not solve the problem.
```

---

### Exercise 6.3 — Replace RNN with LSTM

**Q1.** On short sequences (seq_len=10), the RNN and LSTM perform comparably — there are only 10 gradient multiplications, so vanishing is not an issue. For short sequences, prefer the RNN: fewer parameters, faster to train, equally accurate. Use LSTM when seq_len > ~20.

**Q2.** No. For short sequences the extra parameters add training time without adding accuracy. The ~4× parameter cost of an LSTM is only justified when the task requires long-range memory.

```python
torch.manual_seed(SEED)
X_short, y_short = make_sine_dataset(SHORT)
tr_short, va_short = make_loaders(X_short, y_short)
lstm_short = LSTMNet(input_size=1, hidden_size=32, num_layers=1, dropout=0.0).to(device)
_, lstm_short_val = run_training(
    lstm_short, tr_short, va_short, nn.MSELoss(),
    optim.Adam(lstm_short.parameters(), lr=1e-3),
    num_epochs=40, device=device, verbose_every=None)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rnn_results[f'RNN  seq={SHORT}'], label=f'RNN  seq={SHORT}', color='#27ae60', lw=2)
ax.plot(lstm_short_val,                   label=f'LSTM seq={SHORT}', color='#2980b9', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val MSE'); ax.legend()
plt.tight_layout(); plt.show()
# Both converge similarly — LSTM advantage only appears on long sequences.
```

---

### Exercise 6.4 — End-to-End CO₂ Forecasting

**Task A — shorter window (WINDOW_SIZE=26):**  
MAE typically worsens with a 26-week window. The yearly seasonal cycle takes 52 weeks to complete. A 26-week window only sees half a cycle — the model cannot observe a full summer dip followed by a winter rise in a single input window, so it learns the pattern less precisely.

```python
WINDOW_SIZE_A = 26
X_a, y_a = make_windows(scaled, WINDOW_SIZE_A)
tr_split_a = split - WINDOW_SIZE_A
X_tr_a, y_tr_a = X_a[:tr_split_a], y_a[:tr_split_a]
X_va_a, y_va_a = X_a[tr_split_a:], y_a[tr_split_a:]
# ... (same training loop; compare final MAE)
```

**Task B — smaller model (hidden_size=32):**  
MAE increases moderately. With only 32 hidden units the model has less capacity to represent the combined trend + seasonality. In practice, MAPE above ~1% on the CO₂ series suggests the model is missing systematic patterns.

**Q1 — Why is a random split wrong?**  
A random split places some future observations in the training set. The model sees the future during training — this is data leakage. At deployment no future data exists. A temporal split honestly simulates the real-world scenario: train on everything before time $t$, evaluate on everything after.

**Q2 — Why is MAPE useful here?**  
MAPE expresses error as a percentage of the actual value, making it scale-independent. This is useful when comparing forecast quality across different series (e.g., a series ranging 300–370 vs one ranging 0–100). For CO₂, MAPE tells you what fraction of the typical reading you are missing — intuitive for both scientists and policymakers.

---

### Exercise 6.5 — Pipeline for Sequence Models

**Task A — trigger a validation error:**
```python
v1 = ModelPipeline.load('co2_forecast_v1.pth')
bad_input = torch.zeros(1, 30, 1)   # wrong: model expects window_size=52
try:
    v1.validate_input(bad_input)
except ValueError as e:
    print(f'Caught expected error: {e}')
# Output: Window size mismatch: model expects 52, got 30.
```

**Q1 — Why store window_size inside model_config?**  
A separate variable in your script can easily be changed or forgotten. Storing it inside the `.pth` file means anyone loading the model on any machine automatically has the correct window size — the pipeline is self-contained.

**Q2 — Why does forecast quality degrade over time?**  
After the first few steps, the model is feeding its own (imperfect) predictions back as inputs. Errors compound: a small mistake at step 5 becomes the input at step 6, making step 6 slightly worse, and so on across 52 steps. The further into the future, the more the model is forecasting from its own noise rather than real data.

---

### Exercise 6.6a — FFN Autoencoder

```python
torch.manual_seed(SEED)
X_normal    = torch.randn(10_000, 15)
X_anomalous = torch.randn(10, 15) + 5.0

ae_fraud  = FFNAutoencoder(input_dim=15, bottleneck_dim=5)
crit      = nn.MSELoss()
opt       = optim.Adam(ae_fraud.parameters(), lr=1e-3)
loader    = DataLoader(TensorDataset(X_normal, X_normal), batch_size=256, shuffle=True)

for epoch in range(20):
    for X_b, _ in loader:
        opt.zero_grad()
        loss = crit(ae_fraud(X_b), X_b)
        loss.backward(); opt.step()

ae_fraud.eval()
with torch.no_grad():
    err_normal    = ((ae_fraud(X_normal)    - X_normal)    ** 2).mean(dim=1)
    err_anomalous = ((ae_fraud(X_anomalous) - X_anomalous) ** 2).mean(dim=1)

print(f'Mean error — normal     : {err_normal.mean():.4f}')
print(f'Mean error — anomalous  : {err_anomalous.mean():.4f}')
# Anomalous error should be much higher — the model has never seen
# data from this distribution, so it cannot reconstruct it well.
```

---

### Exercise 6.6b — CNN Autoencoder

```python
class CNNAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 16, 2, stride=2), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 2, stride=2), nn.Sigmoid(),
        )
    def forward(self, x): return self.decoder(self.encoder(x))

cnn_ae   = CNNAutoencoder()
x_img    = torch.randn(2, 1, 28, 28)
x_hat    = cnn_ae(x_img)
print(f'Input: {tuple(x_img.shape)}  Output: {tuple(x_hat.shape)}')  # (2,1,28,28)
print(f'CNN AE parameters : {sum(p.numel() for p in cnn_ae.parameters()):,}')
ffn_flat = FFNAutoencoder(input_dim=784, bottleneck_dim=64)
print(f'FFN AE parameters : {sum(p.numel() for p in ffn_flat.parameters()):,}')
# CNN has far fewer parameters because kernels are shared across all spatial positions.
# An FFN has a separate weight for every pixel-to-neuron connection.
```

---

### Exercise 6.6c — LSTM Autoencoder

```python
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, seq_len=52):
        super().__init__()
        self.seq_len      = seq_len
        self.encoder_lstm = nn.LSTM(input_size,  hidden_size, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        _, (h_n, c_n)   = self.encoder_lstm(x)
        decoder_input   = h_n[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _      = self.decoder_lstm(decoder_input, (h_n, c_n))
        return self.output_layer(decoded)

lstm_ae = LSTMAutoencoder(input_size=1, hidden_size=32, seq_len=52)
x_seq   = torch.randn(4, 52, 1)
x_hat   = lstm_ae(x_seq)
print(f'Input: {tuple(x_seq.shape)}  Output: {tuple(x_hat.shape)}')

# Training (one batch):
criterion = nn.MSELoss()
loss = criterion(x_hat, x_seq)   # target IS the input
print(f'Reconstruction loss (untrained): {loss.item():.4f}')

# Anomaly detection:
recon_err = ((lstm_ae(x_seq) - x_seq) ** 2).mean(dim=[1, 2])
threshold = recon_err.mean() + 2 * recon_err.std()
anomalies = (recon_err > threshold).nonzero(as_tuple=True)[0]
print(f'Anomalous sample indices: {anomalies.tolist()}')
```


---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*